# MAGFiLO MaskDINO (detectron2, ResNet-50)

Add Data → `filament-segmentation-2026`, GPU on.

**Перед запуском:** закоммить и запушь ветку `feature/mask-dino-dev` (ноутбук клонирует её с GitHub).

Порядок: установка → `prepare_data` → smoke → полное обучение.

detectron2 ставим в **системный Python** (не в uv-venv): только там предустановлен GPU-торч Kaggle.

In [ ]:
import os, sys
from pathlib import Path

try:
    from kaggle_secrets import UserSecretsClient
    sec = UserSecretsClient()
    for name in ("GIT_URL", "GITHUB_TOKEN", "DAGSHUB_TOKEN"):
        try:
            os.environ[name] = sec.get_secret(name)
        except Exception:
            pass
except Exception:
    pass

REPO = Path("/kaggle/working/filament-segmentation-2026")
# для приватного репо положи аутентифицированный URL в Kaggle Secret GIT_URL
url = os.environ.get("GIT_URL") or "https://github.com/d1maal/filament-segmentation-2026"
branch = os.environ.get("GIT_BRANCH") or "feature/mask-dino-dev"

if not (REPO / "run.py").exists():
    !git clone --depth 1 --branch {branch} {url} {REPO}

if (REPO / "run.py").exists():
    root = REPO
else:
    candidates = list(Path("/kaggle/input").rglob("run.py"))
    if not candidates:
        raise FileNotFoundError(
            "Репо не найдено: задай GIT_URL или загрузи репо как Kaggle Dataset"
        )
    root = candidates[0].parent

sys.path.insert(0, str(root))
os.chdir(root)
print("repo:", root)

import torch
print(
    "torch:", torch.__version__,
    "| cuda:", torch.version.cuda,
    "| gpu:", torch.cuda.is_available(),
)

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
!pip install -q git+https://github.com/facebookresearch/detectron2.git

In [ ]:
import os

# CUDA-ядро MSDeformAttn
os.environ.setdefault("CUDA_HOME", "/usr/local/cuda")
ops = "src/models/mask_dino_dtn/maskdino/modeling/pixel_decoder/ops"

In [ ]:
!cd {ops} && bash make.sh

In [ ]:
import MultiScaleDeformableAttention
import detectron2
print("detectron2 + MultiScaleDeformableAttention OK")

In [ ]:
!python -m src.models.mask_dino_dtn.prepare_data

In [ ]:
!python run.py check

In [ ]:
!python run.py mask_dino_dtn train --smoke --run-name smoke

In [ ]:
# !python run.py mask_dino_dtn train --size 512 --epochs 8 --run-name mask_dino_dtn_run